# Get to Know a Dataset: CiteSeer Document Repository
This notebook serves as a guided tour of the [CiteSeer Document Repository] dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).


### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.


At the top level of our S3 bucket, the dataset has directories which contains:
 1. data 
 2. docs
 3. metadata
 4. README.txt

The data directory contains a nested structure of sub directories (or prefixes) which at the leaf contains a single pdf file.

Example:
s3://citeseer-repository/data/00/00/2a/b7/88/20/0300002ab788204c8fd919ef037e06e6918ea5bc/0300002ab788204c8fd919ef037e06e6918ea5bc.pdf


First we will import the Python libraries required throughout this notebook.

In [1]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
%pip install --no-cache-dir \
    boto3 \
    pypdf \
    "gliner==0.2.28" \
    "datasets>=2.19.0" \
    "packaging>=23.2.0,<26.0.0"
# Import the libraries required for this notebook
# Built-ins

%pip install --ignore-installed --no-deps tensorflow==2.19.1
%pip uninstall -y tensorflow tf-keras keras

import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

Note: you may need to restart the kernel to use updated packages.
  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (645.0 MB)
Note: you may need to restart the kernel to use updated packages.
Found existing installation: tensorflow 2.19.1
Uninstalling tensorflow-2.19.1:
  Successfully uninstalled tensorflow-2.19.1
Note: you may need to restart the kernel to use updated packages.


Consuming the dataset

In [2]:
import boto3

bucket = "citeseerx-repository-732778637517-us-east-2-an"
s3 = boto3.client('s3')

# Querying with 'data/' as the prefix and '/' as the delimiter
response = s3.list_objects_v2(Bucket=bucket,  Delimiter='/')

if 'CommonPrefixes' in response:
    for item in response['CommonPrefixes']:
        print(item['Prefix'])  # This will print: data/00/, data/01/, etc.
else:
    print("No prefixes found.")

# Each top level prefix contains around 600 GB of data

data/
docs/
metadata/


<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Our dataset consists of PDF documents as crawled from the web, by a crawler focused on academic documents such as papers, books, articles. The format of the document is PDF as this was the original format the document was found on the internet. Some of the documents were also crawled as PS, which we have converted to PDF.

<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

The following cells uses boto3 to stream the PDF directly from S3 into memory, pypdf to extract the text without saving files locally, and gliner (a highly flexible Generalist Model for Named Entity Recognition) to pull out titles, authors you need.

We will start by initializing the libraries required for NER

In [3]:
import io
import boto3
from pypdf import PdfReader
from gliner import GLiNER

# Initialize the GLiNER model
print("Loading GLiNER model...")
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
print("Model loaded successfully.")

2026-09-26 06:15:18.970958056 [W:onnxruntime:Default, device_discovery.cc:285 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:94 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Loading GLiNER model...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Model loaded successfully.


This cell defines the logic to fetch the PDF directly into an in-memory byte stream and parse out its text content.

In [4]:
def extract_text_from_s3_pdf(s3_url):
    """
    Downloads a PDF from an S3 URL into memory and extracts its text.
    Format expected: s3://bucket-name/path/to/file.pdf
    """
    # Parse bucket and key from the S3 URL
    if not s3_url.startswith("s3://"):
        raise ValueError("URL must start with 's3://'")
    path_parts = s3_url.replace("s3://", "").split("/", 1)
    bucket_name = path_parts[0]
    key = path_parts[1]
    # Initialize S3 client
    s3 = boto3.client('s3')
    print(f"Fetching PDF from bucket: {bucket_name}")
    # Fetch object into memory buffer
    response = s3.get_object(Bucket=bucket_name, Key=key)
    pdf_file_bytes = response['Body'].read()
    # Read PDF text using pypdf
    print("Extracting text from PDF...")
    pdf_stream = io.BytesIO(pdf_file_bytes)
    reader = PdfReader(pdf_stream)
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"
    return full_text.strip()

GLiNER allows you to define arbitrary entity labels on the fly. We'll look for person and title, but you can expand this array to catch other fields (like organization or date).

In [5]:
# 1. Define your S3 target file
s3_pdf_path = "s3://citeseerx-repository-732778637517-us-east-2-an/data/00/00/00/88/bb/c1/5a/00000088bbc15a03ab89d8da6c356bf25aea9519/00000088bbc15a03ab89d8da6c356bf25aea9519.pdf"

try:
    # 2. Extract text from the PDF via S3
    document_text = extract_text_from_s3_pdf(s3_pdf_path)
    # GLiNER works best on a per-paragraph or chunked basis.
    # For a high-level metadata pull, the first ~2000 characters typically hold titles and author names.
    sample_chunk = document_text[:2000]
    # 3. Define the custom labels you want GLiNER to search for
    labels = ["person", "title"]
    # 4. Predict entities
    entities = model.predict_entities(sample_chunk, labels, threshold=0.4)
    # 5. Display the structured results
    print("\n=== Extracted Content ===")
    for entity in entities:
        print(f"[{entity['label'].upper()}]: {entity['text']} (Confidence: {entity['score']:.2f})")
except Exception as e:
    print(f"An error occurred: {e}")


Fetching PDF from bucket: citeseerx-repository-732778637517-us-east-2-an
Extracting text from PDF...


/opt/conda/lib/python3.12/site-packages/gliner/data_processing/processor.py:422: UserWarning: Sentence of length 389 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



=== Extracted Content ===
[PERSON]: Mostafa Adimy (Confidence: 0.98)
[PERSON]: Fabien Crauste (Confidence: 0.96)
[PERSON]: Shigui Ruan (Confidence: 0.87)
[PERSON]: mostafa.adimy (Confidence: 0.55)


<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

We have been able to construct large scale citation graphs with these documents. By extracting the titles, author names, citations and their context, we have been able to identify highly impactful authors, publications across several domains.

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?
There are several questions on citations, collaborations, contexts and several aspects of the academic document itself which have been actively been studied and published. A significant number of these publications outline future work which can be researched, implemented and lead to further discoveries.